In [33]:
!pip install --upgrade transformers

In [34]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd
import json
import re

## ПРОВЕРИМ ТОЧНОСТЬ НА ТЕСТОВОМ ДАТАСЕТЕ БЕЗ ДООБУЧЕНИЯ

In [35]:
data_2000 = pd.read_csv("df_2000_all_new.csv")
data_2000

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,доходы населения,валютный курс,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text
0,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1,0,"""Индекс Мосбиржи -3%. Это самое большое дневно..."
1,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,2,0,"""​​Циан взлетел на апелляции Крупнейший в Росс..."
2,not stated,down,up,not stated,not stated,not stated,down,not stated,not stated,not stated,...,down,not stated,not stated,down,not stated,not stated,up,2,1,"""С 5 февраля ЕС вводит эмбарго на российские п..."
3,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,up,3,0,Госбюджет РФ рискует недополучить еще больше н...
4,not stated,not stated,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,4,0,"""Новый отчёт Сбера (SBER): чистая прибыль за 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1801,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1488,0,• Объем торгов металлами в феврале составил 61...
1802,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1489,0,"""​​Страхованию инвестиций на ИИС — быть В дека..."
1803,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1489,1,"""​​Что означает 100 рублей за доллар для префо..."
1804,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1491,0,В начале октября валютный курс тестирует сопро...


In [36]:
## ТУТ ТОЖЕ КОСЯКА НЕ БЫЛО)
data_2000 = data_2000[data_2000['doc_id'] != 'up']
data_2000['doc_id'] = data_2000['doc_id'].astype('int')

test = data_2000[(data_2000['doc_id'] > 1234)]

/tmp/ipykernel_664/733867683.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_2000['doc_id'] = data_2000['doc_id'].astype('int')


In [37]:
dataset = list(test['original_text'])

## ТЕСТ МОДЕЛИ БЕЗ ОБУЧЕНИЯ

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Загрузка токенизатора и модели
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16, # или torch.float16
    device_map="auto"           # автоматически распределит веса на GPU
)

responces = []
with torch.no_grad():
    for i, text in enumerate(dataset, 1):
        print(f'Обработка документа {i}...')


        prompt = ( "Твоя задача — анализировать новости "
        "и определять динамику 18 показателей. Оценивай прямые упоминания и логические следствия. "
        "Допустимые значения изменений ТОЛЬКО ТАКИЕ: 'up', 'down', 'not stated'. "
        "Ответь строго в формате JSON без вступлений и пояснений: "
        "{"
        "\"процентная ставка\": \"тип изменения\","
        "\"ВВП\": \"тип изменения\","
        "\"инфляция\": \"тип изменения\","
        "\"безработица\": \"тип изменения\","
        "\"капитал\": \"тип изменения\","
        "\"инвестиции\": \"тип изменения\","
        "\"производство\": \"тип изменения\","
        "\"потребление\": \"тип изменения\","
        "\"численность рабочей силы\": \"тип изменения\","
        "\"сбережения\": \"тип изменения\","
        "\"заработные платы\": \"тип изменения\","
        "\"доходы населения\": \"тип изменения\","
        "\"валютный курс\": \"тип изменения\","
        "\"импорт\": \"тип изменения\","
        "\"экспорт\": \"тип изменения\","
        "\"государственные расходы\": \"тип изменения\","
        "\"государственный долг\": \"тип изменения\","
        "\"дефицит бюджета\": \"тип изменения\""
        "} "
        f"Текст новости: {text}"
    )
        messages = [
            {"role": "system", "content": "Ты — экспертный макроэкономический аналитик."},
            {"role": "user", "content": prompt}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        responces.append(response)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Обработка документа 1...
Обработка документа 2...
Обработка документа 3...
Обработка документа 4...
Обработка документа 5...
Обработка документа 6...
Обработка документа 7...
Обработка документа 8...
Обработка документа 9...
Обработка документа 10...
Обработка документа 11...
Обработка документа 12...
Обработка документа 13...
Обработка документа 14...
Обработка документа 15...
Обработка документа 16...
Обработка документа 17...
Обработка документа 18...
Обработка документа 19...
Обработка документа 20...
Обработка документа 21...
Обработка документа 22...
Обработка документа 23...
Обработка документа 24...
Обработка документа 25...
Обработка документа 26...
Обработка документа 27...
Обработка документа 28...
Обработка документа 29...
Обработка документа 30...
Обработка документа 31...
Обработка документа 32...
Обработка документа 33...
Обработка документа 34...
Обработка документа 35...
Обработка документа 36...
Обработка документа 37...
Обработка документа 38...
Обработка документа 3

In [ ]:
## СОХРАРНИМ РЕЗУЛЬТАТЫ НЕ ОБУЧЕННОГО КВЕНА
df_qwen_untr = [{'responce': responces[i], 'doc_id': test.iloc[i, -3],
      'chunk_id': test.iloc[i, -2], 'original_text': test.iloc[i, -1]} for i in range(len(responces))]
df_qwen_untr = pd.DataFrame(df_qwen_untr)
df_qwen_untr

In [ ]:
df_qwen_untr.to_csv('df_qwen_untr_test.csv', index=False)

In [ ]:
df_to_check = test
# ------------------------------------------

expected_keys = [
    "процентная ставка", "ВВП", "инфляция", "безработица", "капитал",
    "инвестиции", "производство", "потребление", "численность рабочей силы",
    "сбережения", "заработные платы", "доходы населения", "валютный курс",
    "импорт", "экспорт", "государственные расходы", "государственный долг",
    "дефицит бюджета"
]

total_accuracy = 0
valid_json_count = 0
k = len(responces) # Берем по количеству реально полученных ответов

print("--- Начинаю робастный расчет метрик ---")

for i, pred_raw in enumerate(responces):
    try:
        # 1. Очистка от мусора (markdown, пояснения)
        clean_pred = pred_raw.strip()
        json_match = re.search(r'\{.*\}', clean_pred, re.DOTALL)

        if not json_match:
            print(f"Документ {i+1}: JSON не найден в тексте")
            continue

        pred_json = json.loads(json_match.group())

        # 2. Сравнение
        correct_in_doc = 0
        for entity in expected_keys:
            # Получаем ответ модели
            model_val = str(pred_json.get(entity, "missing")).lower().strip()

            # Получаем правильный ответ из твоего датафрейма
            # Используем .iloc[i], так как мы шли по порядку
            true_val = str(df_to_check.iloc[i][entity]).lower().strip()

            if model_val == true_val:
                correct_in_doc += 1
            # Если хочешь видеть, где модель косячит, расскомментируй:
            # else:
            #     print(f"Док {i+1} | {entity}: модель={model_val}, факт={true_val}")

        total_accuracy += (correct_in_doc / len(expected_keys))
        valid_json_count += 1

    except json.JSONDecodeError:
        print(f"Документ {i+1}: Ошибка декодирования JSON")
    except KeyError as e:
        print(f"Документ {i+1}: В датафрейме нет колонки {e}")
    except Exception as e:
        print(f"Документ {i+1}: Непредвиденная ошибка {type(e).__name__}: {e}")

final_accuracy = total_accuracy / k if k > 0 else 0

print("-" * 30)
print(f"ИТОГОВАЯ ACCURACY: {final_accuracy:.4f}")
print(f"Успешно обработано: {valid_json_count} из {k}")

## КАЧЕСТВО УЖЕ НЕПЛОХОЕ 92%, НО МЫ ХОТИМ ЛУЧШЕ

---



